### Alkuperäisen datan käsittelyä


In [70]:
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)

# lets print multiple outputs in cell, not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [71]:
# load data
df = pd.read_csv('./data/covid-data.csv')
df

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-03,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-04,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-05,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-06,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-07,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358641,ZWE,Africa,Zimbabwe,2023-11-18,265890.0,0.0,0.0,5725.0,0.0,0.0,...,30.7,36.791,1.7,61.49,0.571,16320539.0,NaN,NaN,NaN,NaN
358642,ZWE,Africa,Zimbabwe,2023-11-19,265890.0,0.0,0.0,5725.0,0.0,0.0,...,30.7,36.791,1.7,61.49,0.571,16320539.0,NaN,NaN,NaN,NaN
358643,ZWE,Africa,Zimbabwe,2023-11-20,265890.0,0.0,0.0,5725.0,0.0,0.0,...,30.7,36.791,1.7,61.49,0.571,16320539.0,NaN,NaN,NaN,NaN
358644,ZWE,Africa,Zimbabwe,2023-11-21,265890.0,0.0,0.0,5725.0,0.0,0.0,...,30.7,36.791,1.7,61.49,0.571,16320539.0,NaN,NaN,NaN,NaN


In [72]:
# make mask data (from https://www.cureus.com/articles/93826-correlation-between-mask-compliance-and-covid-19-outcomes-in-europe#!/)
mask_data = {
    'location': ['Albania', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia', 'Hungary', 'North Macedonia', 'Poland', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Belarus', 'Estonia', 'Latvia', 'Lithuania', 'Republic of Moldova', 'Ukraine', 'Austria', 'Belgium', 'Denmark', 'Finland', 'France', 'Germany', 'Greece', 'Ireland', 'Italy', 'Netherlands', 'Norway', 'Portugal', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom', 'Northern Ireland'],
    'avg_mask_usage': [0.53, 0.40, 0.55, 0.29, 0.52, 0.77, 0.67, 0.72, 0.81, 0.54, 0.76, 0.69, 0.55, 0.64, 0.64, 0.74, 0.66, 0.67, 0.55, 0.71, 0.14, 0.46, 0.76, 0.57, 0.84, 0.71, 0.91, 0.51, 0.29, 0.84, 0.95, 0.05, 0.53, 0.62, 0.68],
}

df_mask = pd.DataFrame(mask_data)

# join mask data to covid data
df_covid = df.merge(df_mask, on='location')

# drop countries, that are not in mask data
df_covid = df_covid.dropna(subset=['avg_mask_usage'])

# sort dataframe by date
df_covid['date'] = pd.to_datetime(df_covid['date']).dt.tz_localize(None)
df_covid = df_covid.sort_values(by='date')
df_covid.to_csv('./data/covid_data_mask.csv')

### Valmiiksi käsitellyn datan muokkaamista
    - kaikille locations samat päivämäärät
    - karsitaan datasta vikat ja ekat pois (number of datehs ei muutu)

In [76]:
prep = pd.read_csv('./preprocessed_covid_data.csv')
#prep

# listaks kaikki päivämäärät per location
setlist = list(prep.groupby('location').agg({'date': lambda x: set(x)}).reset_index()['date'])

# kaikista listoista (tai oikeestaan muutetaan seteiks) intersection
u = list(set.intersection(*setlist))

# karsitaan datasta parilliset päivät
u2 = []
for i in u:
    if i % 20 == 0:
        u2.append(i)

# --> locataan ne päivät dataframesta!
prep2 = prep.loc[prep['date'].isin(u2)].drop(columns=['Unnamed: 0'])
prep2.to_csv('./data/intersection_preprocessed_covid_data.csv')
prep2

# poistetaan randomilla vielä nää maat tuolta datasta
category = prep2.groupby('mask_category')['location'].apply(lambda x: list(x.unique())).reset_index()


,location,date,total_deaths_per_million,mask_category
1081,Slovakia,100,0.354,3
1082,Finland,100,43.496,1
1083,Serbia,100,12.370,1
1084,Ireland,100,109.096,2
1085,Lithuania,100,5.818,3
...,...,...,...,...
44009,Denmark,1400,1500.104,1
44010,Hungary,1400,4899.820,3
44011,Ukraine,1400,2768.594,2
44012,Norway,1400,1054.777,1


In [77]:
# kaikilla oli 43 791 rivii ja parillisilla 21 879 rivii

def flatten(l):
    return [item for sublist in l for item in sublist]

c1 = set(flatten(category.iloc[[0]]['location'].tolist()))
c2 = set(flatten(category.iloc[[1]]['location'].tolist()))
c3 = set(flatten(category.iloc[[2]]['location'].tolist()))

# poistetaan c1:stä intersection 1 ja 2
i1 = set.intersection(c1, c2)
# poistetaan c2:stä intersection 2 ja 3
i2 = set.intersection(c2, c3)

ca1 = list(c1 - i1)
ca2 = list(c2 - i2)
ca3 = list(c3)

# valitaan jokaisesta kategoriasta 6 ekaa maata
ca1 = ca1[:4]
ca2 = ca2[:4]
ca3 = ca3[:4]

allca = flatten([ca1, ca2, ca3])

# valitaan nämä maat ja korjataan niiden maski kategoriat
prep3 = pd.DataFrame(prep2)
prep3['mask_category'] = prep3['mask_category'].mask(prep3['location'].isin(ca1), 1)
prep3['mask_category'] = prep3['mask_category'].mask(prep3['location'].isin(ca2), 2)
prep3['mask_category'] = prep3['mask_category'].mask(prep3['location'].isin(ca3), 3)
prep3 = prep3.loc[prep3['location'].isin(allca)]

prep3.to_csv('./data/intersection_preprocessed_covid_data.csv')
